<a href="https://colab.research.google.com/github/brightliam20-ops/MACHINE-LEARNING-ASSIGNMENT/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -q torch torchvision gradio

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

# 1. SETUP REPRODUCIBILITY & DEVICE
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. DATA PREPROCESSING & LOADERS
# ResNet/VGG pre-trained networks expect a 3-channel input scaled to 224x224
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),  # Adapts grayscale MNIST to 3 channels
    transforms.ToTensor(),
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# 3. DEFINE PRE-TRAINED MODEL (Fine-tuning ResNet18)
model = models.resnet18(pretrained=True)

# Freeze early layers if you only want to train the final classifier (optional but recommended)
for param in model.parameters():
    param.requires_grad = False

# Replace the final linear layer to output 10 classes instead of 1,000
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)
model = model.to(device)

# 4. LOSS & OPTIMIZER
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

# 5. TRAINING LOOP (Fine-tune for 2 Epochs)
epochs = 2
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = 100.0 * correct / total
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f} - Accuracy: {epoch_acc:.2f}%")

# 6. SAVE MODEL WEIGHTS
# CRITICAL STEP: Save the state dictionary to deploy it on Hugging Face
torch.save(model.state_dict(), "mnist_resnet18.pth")
print("Model saved successfully as 'mnist_resnet18.pth'!")

Using device: cuda


100%|██████████| 9.91M/9.91M [00:01<00:00, 4.99MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 129kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 13.9MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 156MB/s]


Epoch [1/2] - Loss: 0.3833 - Accuracy: 90.48%
Epoch [2/2] - Loss: 0.1665 - Accuracy: 95.12%
Model saved successfully as 'mnist_resnet18.pth'!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

2. Hugging Face Deployment Structure (app.py)
Create a new Hugging Face Space choosing Gradio as the SDK. Inside your Space repository, upload your mnist_resnet18.pth file and define the runtime files.

requirements.txt

torch
torchvision
gradio
pillow
numpy

### app.py

Create a file named `app.py` in your Hugging Face Space repository with the following content:

In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np

# 1. RE-DEFINE THE NETWORKING ARCHITECTURE
# Make sure to load the model on CPU for Hugging Face Spaces if a GPU is not available or explicitly managed.
model = models.resnet18(pretrained=False)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

# Load the saved state dictionary
model.load_state_dict(torch.load("mnist_resnet18.pth", map_location=torch.device('cpu')))
model.eval() # Set model to evaluation mode

# 2. DEFINE PREPROCESSING TRANSFORMS (Must mirror your training setup)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081))
])

# 3. PREDICTION FUNCTION
def predict_digit(input_data):
    if input_data is None:
        return None

    # Handle the ImageEditor dictionary return type
    if isinstance(input_data, dict):
        # 'composite' extracts the user drawing/manipulation channel
        img = input_data.get("composite") or input_data.get("background")
    else:
        img = input_data

    # Ensure data is converted to a PIL Image
    if isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype('uint8'), 'RGBA' if img.shape[-1] == 4 else 'RGB')

    # Apply transformations and add batch dimension
    img_tensor = transform(img).unsqueeze(0)

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Format the outputs as a dictionary mapping label strings to floats for gr.Label
    return {str(i): float(probabilities[i]) for i in range(10)}

# 4. GRADIO INTERFACE DESIGN
# Using gr.ImageEditor allows webcams, uploads, and a brush tool for virtual sketching.
interface = gr.Interface(
    fn=predict_digit,
    inputs=gr.ImageEditor(
        sources=["upload", "webcam"],
        type="pil",
        image_mode="RGB",
        crop_size=(224, 224),
        label="Draw, Upload or Snap a Photo"
    ),
    outputs=gr.Label(num_top_classes=3, label="Top 3 Predictions"),
    title="MNIST Handwritten Digit Classifier",
    description="Fine-tuned ResNet18 pipeline. Draw a single digit in the canvas or upload an image.",
)

# This block is executed when the app.py is run directly
# For Hugging Face Spaces, the `interface` object will be detected and launched automatically.
if __name__ == "__main__":
    interface.launch()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a0b95da691922bdb26.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
app_py_content = """import gradio as gr
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image, ImageDraw, ImageFont
import numpy as np
import os

# 1. RE-DEFINE THE NETWORKING ARCHITECTURE
# Make sure to load the model on CPU for Hugging Face Spaces if a GPU is not available or explicitly managed.
model = models.resnet18(pretrained=False)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

# Load the saved state dictionary
model.load_state_dict(torch.load("mnist_resnet18.pth", map_location=torch.device('cpu')))
model.eval() # Set model to evaluation mode

# 2. DEFINE PREPROCESSING TRANSFORMS (Must mirror your training setup)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081))
])

# 3. PREDICTION FUNCTION
def predict_digit(input_data):
    if input_data is None:
        return None

    # Handle the ImageEditor dictionary return type
    if isinstance(input_data, dict):
        # 'composite' extracts the user drawing/manipulation channel
        img = input_data.get("composite") or input_data.get("background")
    else:
        img = input_data

    # Ensure data is converted to a PIL Image
    if isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype('uint8'), 'RGBA' if img.shape[-1] == 4 else 'RGB')

    # Apply transformations and add batch dimension
    img_tensor = transform(img).unsqueeze(0)

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Format the outputs as a dictionary mapping label strings to floats for gr.Label
    return {str(i): float(probabilities[i]) for i in range(10)}

# Helper to create example digit images dynamically
def create_digit_image(digit, filename, img_size=(224, 224), font_size=150):
    img = Image.new('RGB', img_size, color = 'black')
    d = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except IOError:
        font = ImageFont.load_default()

    bbox = d.textbbox((0,0), str(digit), font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    x = (img_size[0] - text_width) / 2
    y = (img_size[1] - text_height) / 2

    d.text((x, y), str(digit), fill="white", font=font)
    img.save(filename)

# Create 'examples' directory if it doesn't exist
os.makedirs("examples", exist_ok=True)

# Create example images
create_digit_image(7, "examples/digit_7.png")
create_digit_image(3, "examples/digit_3.png")


# 4. GRADIO INTERFACE DESIGN
# Using gr.ImageEditor allows webcams, uploads, and a brush tool for virtual sketching.
interface = gr.Interface(
    fn=predict_digit,
    inputs=gr.ImageEditor(
        sources=["upload", "webcam"],
        type="pil",
        image_mode="RGB",
        label="Draw, Upload or Snap a Photo"
    ),
    outputs=gr.Label(num_top_classes=3, label="Top 3 Predictions"),
    title="MNIST Handwritten Digit Classifier",
    description="Fine-tuned ResNet18 pipeline. Draw a single digit in the canvas or upload an image.",
    examples=["examples/digit_7.png", "examples/digit_3.png"]
)

# This block is executed when the app.py is run directly
# For Hugging Face Spaces, the `interface` object will be detected and launched automatically.
if __name__ == "__main__":
    interface.launch()"""

with open('app.py', 'w') as f:
    f.write(app_py_content)

print("app.py has been updated with embedded example creation.")

app.py has been updated with embedded example creation.


In [ ]:
requirements_content = """torch
torchvision
gradio
pillow
numpy"""

with open('requirements.txt', 'w') as f:
    f.write(requirements_content)

print("requirements.txt has been created.")

requirements.txt has been created.


In [ ]:
import gradio as gr
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np

# 1. RE-DEFINE THE NETWORKING ARCHITECTURE
model = models.resnet18(pretrained=False)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 10)

# Load the saved state dictionary
model.load_state_dict(torch.load("mnist_resnet18.pth", map_location=torch.device('cpu')))
model.eval()

# 2. DEFINE PREPROCESSING TRANSFORMS (Must mirror your training setup)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.1307, 0.1307, 0.1307), (0.3081, 0.3081, 0.3081))
])

# 3. PREDICTION FUNCTION
def predict_digit(input_data):
    if input_data is None:
        return None

    # Handle the ImageEditor dictionary return type
    if isinstance(input_data, dict):
        # 'composite' extracts the user drawing/manipulation channel
        img = input_data.get("composite") or input_data.get("background")
    else:
        img = input_data

    # Ensure data is converted to a PIL Image
    if isinstance(img, np.ndarray):
        img = Image.fromarray(img.astype('uint8'), 'RGBA' if img.shape[-1] == 4 else 'RGB')

    # Apply transformations and add batch dimension
    img_tensor = transform(img).unsqueeze(0)

    with torch.no_grad():
        outputs = model(img_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Format the outputs as a dictionary mapping label strings to floats for gr.Label
    return {str(i): float(probabilities[i]) for i in range(10)}

# 4. GRADIO INTERFACE DESIGN
# Using gr.ImageEditor allows webcams, uploads, and a brush tool for virtual sketching.
interface = gr.Interface(
    fn=predict_digit,
    inputs=gr.ImageEditor(
        sources=["upload", "webcam"],
        type="pil",
        image_mode="RGB",
        crop_size=(224, 224),
        label="Draw, Upload or Snap a Photo"
    ),
    outputs=gr.Label(num_top_classes=3, label="Top 3 Predictions"),
    title="MNIST Handwritten Digit Classifier",
    description="Fine-tuned ResNet18 pipeline. Draw a single digit in the canvas or upload an image.",
)

if __name__ == "__main__":
    interface.launch()

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://af9c5cc8e958d89b05.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!ls -l

total 12
-rw-r--r-- 1 root root 2649 Jun  8 08:46 app.py
-rw-r--r-- 1 root root 2557 Jun  8 08:44 app.py.py
drwxr-xr-x 1 root root 4096 Jun  4 13:39 sample_data


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import os

def create_digit_image(digit, filename, img_size=(224, 224), font_size=150):
    img = Image.new('RGB', img_size, color = 'black')
    d = ImageDraw.Draw(img)
    try:
        # Try to use a common font
        font = ImageFont.truetype("arial.ttf", font_size)
    except IOError:
        # Fallback to default font if arial.ttf is not found
        font = ImageFont.load_default()

    # Calculate text position to center it
    bbox = d.textbbox((0,0), str(digit), font=font)
    text_width = bbox[2] - bbox[0]
    text_height = bbox[3] - bbox[1]

    x = (img_size[0] - text_width) / 2
    y = (img_size[1] - text_height) / 2

    d.text((x, y), str(digit), fill="white", font=font)
    img.save(filename)

# Create 'examples' directory if it doesn't exist
os.makedirs("examples", exist_ok=True)

# Create example images
create_digit_image(7, "examples/digit_7.png")
create_digit_image(3, "examples/digit_3.png")

print("Example images 'digit_7.png' and 'digit_3.png' created in the 'examples' directory.")

Example images 'digit_7.png' and 'digit_3.png' created in the 'examples' directory.
